# Local Photo ONNX Fixture Test

Upload a photo from your laptop, run the exported hallway ONNX model, detect lighting fixtures, and inspect lux:

- directly under each detected fixture
- at the midpoint between adjacent fixtures

This notebook uses the same ONNX-style preprocessing as the Raspberry Pi prototype and saves a compact overlay plus a JSON summary under `hallway_lighting_runs/local_photo_tests/`.


In [2]:
%pip install onnxruntime pillow matplotlib ipywidgets numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.6 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Arwin-K/APS112-DL-Lighting-Camera.git"
COLAB_REPO_DIR = Path("/content/APS112-DL-Lighting-Camera")

if not COLAB_REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
else:
    print(f"Using existing runtime clone: {COLAB_REPO_DIR}")


In [4]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
import importlib.util
import json
import sys
from typing import Any

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
from PIL import Image


PROJECT_ROOT_OVERRIDE: Path | None = None
MODEL_FILENAME = "hallway_multitask_unet_drive_prototype.onnx"
COLAB_PROJECT_ROOT = Path("/content/APS112-DL-Lighting-Camera/hallway_lighting")


def discover_project_root() -> Path:
    candidates: list[Path] = []
    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(PROJECT_ROOT_OVERRIDE.expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        COLAB_PROJECT_ROOT,
        cwd,
        cwd / "hallway_lighting",
    ])

    current = cwd
    for _ in range(6):
        candidates.append(current)
        candidates.append(current / "hallway_lighting")
        if current.parent == current:
            break
        current = current.parent

    seen: set[Path] = set()
    ordered_candidates: list[Path] = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        if resolved not in seen:
            seen.add(resolved)
            ordered_candidates.append(resolved)

    for candidate in ordered_candidates:
        model_path = candidate / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
        fixture_path = candidate / "hallway_lighting" / "utils" / "fixture_detection.py"
        if model_path.exists() and fixture_path.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the hallway_lighting project folder and exported ONNX model. "
        "If you are in Colab, run the clone cell first. Otherwise open the notebook from inside the repo. "
        f"Candidates checked: {[str(path) for path in ordered_candidates]}"
    )


PROJECT_ROOT = discover_project_root()
DEFAULT_ONNX_PATH = PROJECT_ROOT / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
OUTPUT_DIR = PROJECT_ROOT / "hallway_lighting_runs" / "local_photo_tests"
FIXTURE_DETECTION_PATH = PROJECT_ROOT / "hallway_lighting" / "utils" / "fixture_detection.py"


def load_fixture_detection_module():
    spec = importlib.util.spec_from_file_location("fixture_detection_local", FIXTURE_DETECTION_PATH)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load fixture detection helper from {FIXTURE_DETECTION_PATH}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


fixture_detection = load_fixture_detection_module()


def extract_single_map(value: np.ndarray | float | int | None) -> np.ndarray | None:
    if value is None or isinstance(value, (float, int)):
        return None
    array = np.asarray(value)
    if array.ndim == 4:
        array = array[0]
    if array.ndim == 3 and array.shape[0] in {1, 3}:
        array = np.transpose(array, (1, 2, 0))
    if array.ndim == 3 and array.shape[-1] == 1:
        array = array[..., 0]
    return array.astype(np.float32, copy=False)


def extract_scalar(value: np.ndarray | float | int | None, fallback: float = 0.0) -> float:
    if value is None:
        return fallback
    if isinstance(value, (float, int)):
        return float(value)
    array = np.asarray(value).reshape(-1)
    return fallback if array.size == 0 else float(array[0])


def sample_bilinear(map_2d: np.ndarray, x_normalized: float, y_normalized: float) -> float:
    height, width = map_2d.shape
    x = float(np.clip(x_normalized, 0.0, 1.0)) * (width - 1)
    y = float(np.clip(y_normalized, 0.0, 1.0)) * (height - 1)
    x0 = int(np.floor(x))
    x1 = min(x0 + 1, width - 1)
    y0 = int(np.floor(y))
    y1 = min(y0 + 1, height - 1)
    wx = x - x0
    wy = y - y0
    top = (1.0 - wx) * float(map_2d[y0, x0]) + wx * float(map_2d[y0, x1])
    bottom = (1.0 - wx) * float(map_2d[y1, x0]) + wx * float(map_2d[y1, x1])
    return (1.0 - wy) * top + wy * bottom


def sample_point_lux(lux_map: np.ndarray, point_targets: list[Any]) -> dict[str, float]:
    return {
        point.name: sample_bilinear(lux_map, point.x, point.y)
        for point in point_targets
    }


def summarize_lux_map(lux_map: np.ndarray, floor_mask: np.ndarray | None = None) -> dict[str, float]:
    values = np.asarray(lux_map, dtype=np.float32)
    if floor_mask is not None:
        mask = np.asarray(floor_mask).astype(bool)
        if mask.shape == values.shape and float(mask.mean()) > 0.001:
            values = values[mask]
        else:
            values = values.reshape(-1)
    else:
        values = values.reshape(-1)
    if values.size == 0:
        return {"avg_lux": 0.0, "low_lux_p5": 0.0, "high_lux_p95": 0.0}
    return {
        "avg_lux": float(np.mean(values)),
        "low_lux_p5": float(np.percentile(values, 5)),
        "high_lux_p95": float(np.percentile(values, 95)),
    }


def load_onnx_session(onnx_path: Path):
    try:
        import onnxruntime as ort
    except ImportError as exc:
        raise ImportError(
            "onnxruntime is not installed in this notebook environment. Run the setup cell above, restart the kernel, and try again."
        ) from exc

    if not onnx_path.exists():
        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")
    return ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])


def preprocess_uploaded_image(image_bytes: bytes, input_height: int, input_width: int) -> tuple[np.ndarray, np.ndarray]:
    image = Image.open(BytesIO(image_bytes)).convert("RGB")
    resized = image.resize((input_width, input_height), resample=Image.BILINEAR)
    display_rgb = np.asarray(resized).astype(np.float32) / 255.0
    model_input = np.transpose(display_rgb, (2, 0, 1))[None, ...].astype(np.float32)
    return model_input, display_rgb


def build_overlay_figure(display_rgb: np.ndarray, lux_map: np.ndarray, fixture_analysis: dict[str, Any] | None, point_lux: dict[str, float]):
    figure, axis = plt.subplots(1, 1, figsize=(8, 5))
    axis.imshow(display_rgb)
    axis.imshow(lux_map, cmap="inferno", alpha=0.38)
    height, width = display_rgb.shape[:2]

    if fixture_analysis is not None:
        for region in fixture_analysis.get("between_regions", []):
            polygon_points = region.get("polygon") or []
            if polygon_points:
                polygon_pixels = [
                    (float(x_value) * (width - 1), float(y_value) * (height - 1))
                    for x_value, y_value in polygon_points
                ]
                axis.add_patch(
                    Polygon(
                        polygon_pixels,
                        closed=True,
                        facecolor="#80cbc4",
                        edgecolor="#004d40",
                        linewidth=1.2,
                        alpha=0.18,
                    )
                )

        for fixture in fixture_analysis.get("fixtures", []):
            x_pixel = float(fixture["x"]) * (width - 1)
            y_pixel = float(fixture["y"]) * (height - 1)
            axis.scatter(x_pixel, y_pixel, s=120, c="#ffee58", edgecolors="black", linewidths=0.9, zorder=3)
            axis.text(
                x_pixel + 6,
                y_pixel - 6,
                str(fixture["name"]),
                color="white",
                fontsize=8,
                bbox={"facecolor": "black", "alpha": 0.45, "pad": 2},
            )

        for point in fixture_analysis.get("point_targets", []):
            point_name = str(point["name"])
            if point_name not in point_lux:
                continue
            x_pixel = float(point["x"]) * (width - 1)
            y_pixel = float(point["y"]) * (height - 1)
            point_color = "#4fc3f7" if point_name.startswith("between_") else "#ffffff"
            axis.scatter(x_pixel, y_pixel, s=48, c=point_color, edgecolors="black", linewidths=0.8, zorder=4)
            axis.text(
                x_pixel + 4,
                y_pixel + 10,
                f"{point_name}\n{point_lux[point_name]:.1f} lux",
                color="white",
                fontsize=7,
                bbox={"facecolor": "black", "alpha": 0.5, "pad": 2},
                zorder=5,
            )

    axis.set_title("Fixture-aware Lux Overlay")
    axis.axis("off")
    figure.tight_layout()
    return figure


def run_uploaded_photo(image_bytes: bytes, image_name: str, onnx_path: Path, max_fixture_count: int = 8, floor_area_m2: float = 12.0):
    if fixture_detection is None:
        raise RuntimeError(
            "Project root is not configured. Set PROJECT_ROOT_OVERRIDE near the top of the setup cell and rerun it."
        )

    session = load_onnx_session(onnx_path)
    input_shape = session.get_inputs()[0].shape
    input_height = int(input_shape[2])
    input_width = int(input_shape[3])
    model_input, display_rgb = preprocess_uploaded_image(image_bytes, input_height=input_height, input_width=input_width)

    raw_outputs = session.run(None, {session.get_inputs()[0].name: model_input})
    output_names = [
        "lux_map",
        "avg_lux",
        "low_lux_p5",
        "high_lux_p95",
        "floor_mask_pred",
        "albedo_pred",
        "gloss_pred",
        "uncertainty_map",
        "estimated_power_w",
    ]
    named_outputs = {name: value for name, value in zip(output_names, raw_outputs)}

    lux_map = extract_single_map(named_outputs["lux_map"])
    if lux_map is None or lux_map.ndim != 2:
        raise ValueError("Expected a single-channel lux map from the ONNX output.")
    floor_mask = extract_single_map(named_outputs.get("floor_mask_pred"))
    floor_mask_binary = None if floor_mask is None else (np.asarray(floor_mask) > 0.5)

    fixture_layout = fixture_detection.infer_fixture_layout(
        image=display_rgb,
        floor_mask=floor_mask_binary,
        max_fixture_count=max_fixture_count,
        floor_area_m2=floor_area_m2,
    )
    fixture_analysis = None if fixture_layout is None else fixture_layout.to_summary_dict()
    point_targets = [] if fixture_layout is None else list(fixture_layout.point_targets)
    point_lux = sample_point_lux(lux_map, point_targets)
    under_fixture_lux = {name: value for name, value in point_lux.items() if name.startswith("under_")}
    between_fixture_lux = {name: value for name, value in point_lux.items() if name.startswith("between_")}

    lux_summary = summarize_lux_map(lux_map=lux_map, floor_mask=floor_mask_binary)
    figure = build_overlay_figure(display_rgb, lux_map, fixture_analysis, point_lux)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    image_stem = Path(image_name).stem
    overlay_path = OUTPUT_DIR / f"{image_stem}_fixture_lux_overlay.png"
    summary_path = OUTPUT_DIR / f"{image_stem}_fixture_lux_summary.json"
    figure.savefig(overlay_path, bbox_inches="tight")

    result = {
        "image_name": image_name,
        "onnx_path": str(onnx_path),
        "fixture_count": 0 if fixture_analysis is None else int(fixture_analysis.get("inferred_fixture_count", 0)),
        "avg_lux": extract_scalar(named_outputs.get("avg_lux"), fallback=lux_summary["avg_lux"]),
        "low_lux_p5": extract_scalar(named_outputs.get("low_lux_p5"), fallback=lux_summary["low_lux_p5"]),
        "high_lux_p95": extract_scalar(named_outputs.get("high_lux_p95"), fallback=lux_summary["high_lux_p95"]),
        "lux_map_summary": lux_summary,
        "under_fixture_lux": under_fixture_lux,
        "between_fixture_lux": between_fixture_lux,
        "point_lux": point_lux,
        "fixture_analysis": fixture_analysis,
        "overlay_image": str(overlay_path),
        "summary_json": str(summary_path),
    }
    with summary_path.open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)
    return result, figure


print(f"Project root: {PROJECT_ROOT}")
print(f"Default ONNX: {DEFAULT_ONNX_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


Project root: /content/APS112-DL-Lighting-Camera/hallway_lighting
Default ONNX: /content/APS112-DL-Lighting-Camera/hallway_lighting/hallway_lighting_runs/exports/hallway_multitask_unet_drive_prototype.onnx
Output dir: /content/APS112-DL-Lighting-Camera/hallway_lighting/hallway_lighting_runs/local_photo_tests


In [5]:
upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Upload photo")
max_fixtures_widget = widgets.BoundedIntText(value=12, min=1, max=30, description="Max fixtures")
floor_area_widget = widgets.FloatText(value=12.0, description="Floor area m2")
run_button = widgets.Button(description="Run ONNX test", button_style="primary")
output_widget = widgets.Output()


def extract_uploaded_entry(upload_value):
    if isinstance(upload_value, dict) and upload_value:
        first_key = next(iter(upload_value.keys()))
        first_value = upload_value[first_key]
        if isinstance(first_value, dict):
            image_name = first_value.get("name", first_key)
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return str(first_key), first_value

    if isinstance(upload_value, (list, tuple)) and upload_value:
        first_value = upload_value[0]
        if isinstance(first_value, dict):
            image_name = first_value.get("name") or first_value.get("metadata", {}).get("name") or "uploaded_image"
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return "uploaded_image", first_value

    return None, None


def render_result(result: dict[str, Any]):
    display(Markdown(f"**Detected fixtures:** {result['fixture_count']}"))
    display(Markdown(f"**Average lux:** {result['avg_lux']:.1f}"))

    print("Under-fixture lux:")
    if result["under_fixture_lux"]:
        for point_name, lux_value in result["under_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print("Between adjacent fixtures lux:")
    if result["between_fixture_lux"]:
        for point_name, lux_value in result["between_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print(f"Overlay saved to: {result['overlay_image']}")
    print(f"Summary saved to: {result['summary_json']}")


def on_run_clicked(_):
    with output_widget:
        clear_output()
        image_name, image_bytes = extract_uploaded_entry(upload_widget.value)
        if image_bytes is None:
            print("Upload one image first.")
            return

        try:
            result, figure = run_uploaded_photo(
                image_bytes=image_bytes,
                image_name=image_name,
                onnx_path=DEFAULT_ONNX_PATH,
                max_fixture_count=int(max_fixtures_widget.value),
                floor_area_m2=float(floor_area_widget.value),
            )
        except Exception as exc:
            print(f"Inference failed: {exc}")
            return

        render_result(result)
        display(figure)
        plt.close(figure)


run_button.on_click(on_run_clicked)

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Upload a photo and run local ONNX inference</b>"),
            widgets.HTML(f"<code>{DEFAULT_ONNX_PATH}</code>"),
            upload_widget,
            widgets.HBox([max_fixtures_widget, floor_area_widget]),
            run_button,
            output_widget,
        ]
    )
)